In [1]:
import sys
import os
sys.path.append(os.path.abspath('/acne-lds/model'))
sys.path.append(os.path.abspath('model'))
sys.path.append(os.path.abspath("/acne-lds/utils"))

In [2]:
from model_ld_smoothing import AcneModel

/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/io/image.py:11: UserWarning: Failed to load image Python extension: dlopen(/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libpng16.16.dylib
  Referenced from: <5F6B6919-410D-397C-98F2-12C5934F9DBE> /opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/torchvision/image.so
  Reason: tried: '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/Users/malfet/miniforge3/envs/py_39_torch-1.10.2/lib/libpng16.16.dylib' (no such file), '/System/Volu

In [5]:
from predict_on_img import ModelInit
from PIL import Image

model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')
# img = Image.open(PATH_TO_IMAGE)
# predictions = model.predict_on_img(img)

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /Users/utkarshsharma/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth
100%|██████████| 97.8M/97.8M [00:32<00:00, 3.15MB/s]


In [48]:
# Assume ModelInit is imported and your checkpoint path is correct
import torch
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model = model_wrapper.model
model.eval()

# Dummy input tensor (use actual input size expected, e.g. 3x224x224)
example_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)

In [50]:
1/0.225

4.444444444444445

In [52]:
import coremltools as ct

# Convert to CoreML model
coreml_model = ct.convert(
    traced_model,
    inputs=[ct.ImageType(name="input_image",
    shape=(1, 3, 224, 224),  # Adjust to match your model's input dimensions
    bias=[-0.485, -0.456, -0.406],  # ImageNet means for RGB channels 
    scale=1/ 0.2814769, 
    color_layout=ct.colorlayout.RGB)]  # ImageNet std for RGB channels
    )

# Save to file
coreml_model.save("AcneClassification.mlpackage")

When both 'convert_to' and 'minimum_deployment_target' not specified, 'convert_to' is set to "mlprogram" and 'minimum_deployment_target' is set to ct.target.iOS15 (which is same as ct.target.macOS12). Note: the model will not run on systems older than iOS15/macOS12/watchOS8/tvOS15. In order to make your model run on older system, please set the 'minimum_deployment_target' to iOS14/iOS13. Details please see the link: https://apple.github.io/coremltools/docs-guides/source/target-conversion-formats.html
Tuple detected at graph output. This will be flattened in the converted model.
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 181.17 passes/s]


In [35]:
import torch

# Load your model
model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')
model.model.eval()  # Set to evaluation mode

# Create a dummy input that matches what your model expects
# Assuming your input is an RGB image with dimensions [C, H, W]
dummy_input = torch.randn(1, 3, 224, 224)  # Adjust dimensions to match your model's input

# Define the path for the ONNX model
onnx_path = "acne_detection_model.onnx"

# Export the model
torch.onnx.export(
    model.model,                   # The model to export
    dummy_input,                   # Input tensor
    onnx_path,                     # Output file
    export_params=True,            # Export model parameters
    opset_version=12,              # ONNX opset version
    do_constant_folding=True,      # Optimize constant folding
    input_names=["input"],         # Names for inputs
    output_names=["cls", "cou", "cou2cls"],  # Names for outputs
    dynamic_axes={
        "input": {0: "batch_size"},  # Variable batch size
        "cls": {0: "batch_size"},
        "cou": {0: "batch_size"},
        "cou2cls": {0: "batch_size"}
    }
)

print(f"Model exported to {onnx_path}")

Model exported to acne_detection_model.onnx


In [46]:
import coremltools as ct
import torch

# Assuming the traced_model and example_input from previous steps are already set

# Define preprocessing for the model
coreml_model = ct.convert(
    traced_model,
    inputs=[ct.ImageType(name="input_image", 
                         shape=(1, 3, 224, 224),  # Match the model's input shape
    bias=[-0.485, -0.456, -0.406],  # ImageNet means for RGB channels 
    scale=[1/0.229, 1/0.224, 1/0.225],  # ImageNet std for RGB channels
                         color_layout=ct.colorlayout.RGB)]
)

# Save the converted model
coreml_model.save("AcneClassification.mlpackage")


When both 'convert_to' and 'minimum_deployment_target' not specified, 'convert_to' is set to "mlprogram" and 'minimum_deployment_target' is set to ct.target.iOS15 (which is same as ct.target.macOS12). Note: the model will not run on systems older than iOS15/macOS12/watchOS8/tvOS15. In order to make your model run on older system, please set the 'minimum_deployment_target' to iOS14/iOS13. Details please see the link: https://apple.github.io/coremltools/docs-guides/source/target-conversion-formats.html
Tuple detected at graph output. This will be flattened in the converted model.
Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

ERROR - 'mil_backend::insert_image_preprocessing_ops' graph pass produces the following error:

Running MIL backend_mlprogram pipeline:  17%|█▋        | 2/12 [00:00<00:00, 155.58 passes/s]


ValueError: Incompatible dim 3 in shapes (1, 3, 224, 224) vs. (1, 1, 1, 3)

In [33]:

class ModelInit:
    """Class that initialize the model and make prediction on single raw image."""

    def __init__(self, model_type="model_ld_smoothing", path_checkpoint=None, device="cpu"):
        """Init of the object."""
        self.model_type = model_type
        # Create model
        num_acne_cls = 13 if model_type == "model_ld_smoothing" else 4
        self.model = resnet50(num_acne_cls=num_acne_cls)
        # load checkpoint
        checkpoint = torch.load(path_checkpoint, map_location=torch.device(device))
        self.model.load_state_dict(checkpoint["model_state_dict"])
        # transforms
        self.transform = AcneTransformsTorch(train=False)

    def predict_on_img(self, img):
        """Get prediction for given image."""
        self.model.eval()
        with torch.no_grad():
            cls, cou, cou2cls = self.model(self.transform(img)[None, :, :, :])
            # Convert predictions back to Hayashi scale if needed
            if self.model_type == "model_ld_smoothing":
                cls = torch.stack(
                    (
                        torch.sum(cls[:, :1], 1),
                        torch.sum(cls[:, 1:4], 1),
                        torch.sum(cls[:, 4:10], 1),
                        torch.sum(cls[:, 10:], 1),
                    ),
                    1,
                )

            return cls, cou, cou2cls


In [22]:
img = Image.open('../data/Classification/JPEGImages/levle2_165.jpg')

In [9]:
predictions = model.predict_on_img(img)

In [23]:
predictions2 = model.predict_on_img(img)

In [ ]:
import torch
preds_cls = torch.argmax(0.5 * (cls + cou2cls), dim=1)

In [30]:
import torch
preds_cls = torch.argmax(0.5 * (predictions2[0] + predictions2[2]), dim=1)